In [ ]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler

data = pd.read_csv("/content/data.csv")

In [ ]:
data["police_station"].notna().sum()


np.int64(8173)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler

data = pd.read_csv("/content/data.csv")

print(data["priority"].head())

grouped_data = data.groupby("junction").agg(
    event_count=("junction", "size"),
    planned_count=("event_type", lambda x: (x == "planned").sum()),
    unplanned_count=("event_type", lambda x: (x == "unplanned").sum()),
    road_closure_count=("event_type", lambda x: (x == True).sum()),
    high_priority_count=("priority", lambda x: (x == "High").sum())
).reset_index()

display(grouped_data)

features = [
    "event_count",
    "planned_count",
    "unplanned_count",
    "road_closure_count",
    "high_priority_count"
]

X = grouped_data[features].replace([np.inf, -np.inf], np.nan).fillna(0)

kmeans = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

grouped_data["cluster"] = kmeans.fit_predict(X)

display(grouped_data.head())

cluster_summary = grouped_data.groupby("cluster")[features].mean()
display(cluster_summary)

if {"start_datetime", "end_datetime"}.issubset(data.columns):

    data["start_datetime"] = pd.to_datetime(
        data["start_datetime"],
        errors="coerce"
    )

    data["end_datetime"] = pd.to_datetime(
        data["end_datetime"],
        errors="coerce"
    )

    data_cleaned = data.dropna(
        subset=["start_datetime", "end_datetime"]
    ).copy()

    data_cleaned["duration_in_min"] = (
        data_cleaned["end_datetime"]
        - data_cleaned["start_datetime"]
    ).dt.total_seconds() / 60

    avg_duration_per_junction = (
        data_cleaned.groupby("junction")["duration_in_min"]
        .mean()
        .reset_index(name="avg_event_duration")
    )

    most_recent_date = data_cleaned["start_datetime"].max()

    six_months_ago = most_recent_date - pd.DateOffset(months=6)

    recent_events_count = (
        data_cleaned[
            data_cleaned["start_datetime"] >= six_months_ago
        ]
        .groupby("junction")
        .size()
        .reset_index(name="recent_events_last_6_months")
    )

    grouped_data = grouped_data.drop(
        columns=[
            col
            for col in grouped_data.columns
            if col in [
                "avg_event_duration",
                "recent_events_last_6_months"
            ]
        ],
        errors="ignore"
    )

    grouped_data = grouped_data.merge(
        avg_duration_per_junction,
        on="junction",
        how="left"
    )

    grouped_data = grouped_data.merge(
        recent_events_count,
        on="junction",
        how="left"
    )

    grouped_data["avg_event_duration"] = (
        grouped_data["avg_event_duration"].fillna(0)
    )

    grouped_data["recent_events_last_6_months"] = (
        grouped_data["recent_events_last_6_months"].fillna(0)
    )

else:
    grouped_data["avg_event_duration"] = 0
    grouped_data["recent_events_last_6_months"] = 0

display(grouped_data.head())

risk_features = [
    "event_count",
    "road_closure_count",
    "high_priority_count",
    "recent_events_last_6_months"
]

scaler = MinMaxScaler()

grouped_data[risk_features] = scaler.fit_transform(
    grouped_data[risk_features]
)

grouped_data["risk_score"] = (
    0.35 * grouped_data["event_count"]
    + 0.25 * grouped_data["road_closure_count"]
    + 0.20 * grouped_data["high_priority_count"]
    + 0.20 * grouped_data["recent_events_last_6_months"]
)

display(grouped_data)

0    High
1    High
2     Low
3     Low
4     Low
Name: priority, dtype: object


,junction,event_count,planned_count,unplanned_count,road_closure_count,high_priority_count
0,17th Mn 1st Crs Aishwarya Stores Jn,1,0,1,0,0
1,27th Cross Jayanagar(Ganapathi Temple),1,0,1,0,0
2,28thMainJayanagarJunc,6,0,6,0,1
3,29thMainRdBTM LayoutJunc,5,0,5,0,0
4,5thMainHSR,3,1,2,0,3
...,...,...,...,...,...,...
289,YelhankaBypass,19,0,19,0,19
290,YelhankaCircle,34,2,32,0,31
291,Yemalur cross junc,4,0,4,0,4
292,YeshwanthpuraCircle,38,0,38,0,36


,junction,event_count,planned_count,unplanned_count,road_closure_count,high_priority_count,cluster
0,17th Mn 1st Crs Aishwarya Stores Jn,1,0,1,0,0,0
1,27th Cross Jayanagar(Ganapathi Temple),1,0,1,0,0,0
2,28thMainJayanagarJunc,6,0,6,0,1,0
3,29thMainRdBTM LayoutJunc,5,0,5,0,0,0
4,5thMainHSR,3,1,2,0,3,0


,event_count,planned_count,unplanned_count,road_closure_count,high_priority_count
cluster,,,,,
0,4.330189,0.287736,4.042453,0.0,1.844340
1,32.681818,1.363636,31.318182,0.0,31.954545
2,14.550000,0.516667,14.033333,0.0,11.450000


,junction,event_count,planned_count,unplanned_count,road_closure_count,high_priority_count,cluster,avg_event_duration,recent_events_last_6_months
0,17th Mn 1st Crs Aishwarya Stores Jn,1,0,1,0,0,0,0.00000,0.0
1,27th Cross Jayanagar(Ganapathi Temple),1,0,1,0,0,0,0.00000,0.0
2,28thMainJayanagarJunc,6,0,6,0,1,0,0.00000,0.0
3,29thMainRdBTM LayoutJunc,5,0,5,0,0,0,0.00000,0.0
4,5thMainHSR,3,1,2,0,3,0,1.05055,1.0


,junction,event_count,planned_count,unplanned_count,road_closure_count,high_priority_count,cluster,avg_event_duration,recent_events_last_6_months,risk_score
0,17th Mn 1st Crs Aishwarya Stores Jn,0.000000,0,1,0.0,0.000000,0,0.000000,0.000000,0.000000
1,27th Cross Jayanagar(Ganapathi Temple),0.000000,0,1,0.0,0.000000,0,0.000000,0.000000,0.000000
2,28thMainJayanagarJunc,0.079365,0,6,0.0,0.015873,0,0.000000,0.000000,0.030952
3,29thMainRdBTM LayoutJunc,0.063492,0,5,0.0,0.000000,0,0.000000,0.000000,0.022222
4,5thMainHSR,0.031746,1,2,0.0,0.047619,0,1.050550,0.090909,0.038817
...,...,...,...,...,...,...,...,...,...,...
289,YelhankaBypass,0.285714,0,19,0.0,0.301587,2,0.000000,0.000000,0.160317
290,YelhankaCircle,0.523810,2,32,0.0,0.492063,1,-659.922425,0.181818,0.318110
291,Yemalur cross junc,0.047619,0,4,0.0,0.063492,0,0.000000,0.000000,0.029365
292,YeshwanthpuraCircle,0.587302,0,38,0.0,0.571429,1,0.000000,0.000000,0.319841


In [ ]:
num_unique_event_cause = data['event_cause'].nunique()
print(f"Number of unique event causes: {num_unique_event_cause}")

Number of unique event causes: 17


In [ ]:
events_by_address = data.groupby('address')['event_cause'].unique()
print("Number of unique event causes for each address:")
for address, causes in events_by_address.items():
    print(f"Address: {address}\n  Number of unique event causes: {len(causes)}\n")

Number of unique event causes for each address:
Address: "2nd Cross Road, MTB Area, Jayanagar, Bengaluru, Karnataka. Pin-560041 (India)
  Number of unique event causes: 1

Address: 100 Feet Road, Ankanna Reddy Layout, Banaswadi, Bengaluru, Karnataka. Pin-560043 (India)
  Number of unique event causes: 1

Address: 100 Feet Road, BM Sri Circle, Indiranagar, Bengaluru, Karnataka. Pin-560038 (India)
  Number of unique event causes: 1

Address: 100 Feet Road, Defence Colony, Indiranagar, Bengaluru, Karnataka. Pin-560038 (India)
  Number of unique event causes: 1

Address: 100 Feet Road, Jalahalli Cross Junction, Peenya, Bengaluru, Karnataka. Pin-560058 (India)
  Number of unique event causes: 1

Address: 100 Feet Road, Jalahalli Cross, T Dasarahalli, Bengaluru, Karnataka. Pin-560057 (India)
  Number of unique event causes: 2

Address: 100 Feet Road, KFC Circle, Indiranagar, Bengaluru, Karnataka. Pin-560038 (India)
  Number of unique event causes: 1

Address: 100 Feet Road, Kanteerava Studio

In [ ]:
non_null_rows = data.dropna(subset=['start_datetime', 'resolved_datetime', 'event_cause'])
num_non_null_rows = len(non_null_rows)
print(f"Number of data rows with non-null 'start_datetime', 'resolved_datetime', and 'event_cause': {num_non_null_rows}")

Number of data rows with non-null 'start_datetime', 'resolved_datetime', and 'event_cause': 72


In [ ]:
non_null_rows_by_event_type = non_null_rows.groupby('event_cause').size().reset_index(name='count')
print("Number of non-null rows for 'start_datetime', 'resolved_datetime', and 'event_cause' grouped by 'event_type':")
display(non_null_rows_by_event_type)

Number of non-null rows for 'start_datetime', 'resolved_datetime', and 'event_cause' grouped by 'event_type':


,event_cause,count
0,accident,5
1,others,5
2,pot_holes,2
3,tree_fall,2
4,vehicle_breakdown,57
5,water_logging,1


In [ ]:
##### This one is to get the final three csvs

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# =====================================
# LOAD DATA
# =====================================

df = pd.read_csv("/content/data.csv")

# =====================================
# DATETIME PROCESSING
# =====================================

df["start_datetime"] = pd.to_datetime(
    df["start_datetime"],
    errors="coerce"
)

df = df.dropna(subset=["start_datetime"]).copy()

latest_date = df["start_datetime"].max()

# =====================================
# RECENCY WEIGHT
# =====================================

df["days_old"] = (
    latest_date - df["start_datetime"]
).dt.days

HALF_LIFE_DAYS = 90

df["recency_weight"] = np.exp(
    -np.log(2) * df["days_old"] / HALF_LIFE_DAYS
)

# =====================================
# TIME FEATURES
# =====================================

df["hour"] = df["start_datetime"].dt.hour

df["day_type"] = np.where(
    df["start_datetime"].dt.dayofweek >= 5,
    "Weekend",
    "Weekday"
)

# =====================================
# JUNCTION SCORE
# =====================================

junction_stats = df.groupby("junction").agg(

    event_count=(
        "junction",
        "size"
    ),

    road_closure_count=(
        "requires_road_closure",
        lambda x: (
            x.astype(str)
             .str.upper()
             .isin(["TRUE", "YES", "1"])
        ).sum()
    ),

    high_priority_count=(
        "priority",
        lambda x: (
            x.astype(str)
             .str.upper()
             .eq("HIGH")
        ).sum()
    ),

    recency_score=(
        "recency_weight",
        "mean"
    )

).reset_index()

# =====================================
# RECENT EVENTS (6 MONTHS)
# =====================================

six_months_ago = (
    latest_date - pd.DateOffset(months=6)
)

recent_events = (
    df[
        df["start_datetime"] >= six_months_ago
    ]
    .groupby("junction")
    .size()
    .reset_index(
        name="recent_events_last_6_months"
    )
)

junction_stats = junction_stats.merge(
    recent_events,
    on="junction",
    how="left"
)

junction_stats[
    "recent_events_last_6_months"
] = (
    junction_stats[
        "recent_events_last_6_months"
    ].fillna(0)
)

# =====================================
# NORMALIZE FEATURES
# =====================================

risk_features = [

    "event_count",
    "road_closure_count",
    "high_priority_count",
    "recent_events_last_6_months",
    "recency_score"

]

scaler = MinMaxScaler()

junction_stats[risk_features] = (
    scaler.fit_transform(
        junction_stats[risk_features]
    )
)

# =====================================
# FINAL JUNCTION SCORE
# =====================================

junction_stats["risk_score"] = (

    0.30 * junction_stats["event_count"]

    + 0.20 * junction_stats["road_closure_count"]

    + 0.15 * junction_stats["high_priority_count"]

    + 0.20 * junction_stats["recent_events_last_6_months"]

    + 0.15 * junction_stats["recency_score"]

)

junction_stats["risk_score"] = (
    junction_stats["risk_score"] * 100
).round(2)

junction_scores = junction_stats[
    ["junction", "risk_score"]
].sort_values(
    "risk_score",
    ascending=False
)

junction_scores.to_csv(
    "junction_scores.csv",
    index=False
)

print("Saved: junction_scores.csv")

# =====================================
# HOUR MULTIPLIERS
# =====================================

hourly_counts = (

    df.groupby(
        ["junction", "hour"]
    )
    .size()
    .reset_index(
        name="event_count"
    )

)

avg_hourly_counts = (

    hourly_counts
    .groupby("junction")["event_count"]
    .mean()
    .reset_index(
        name="avg_hourly_count"
    )

)

hourly_counts = hourly_counts.merge(
    avg_hourly_counts,
    on="junction"
)

hourly_counts["hour_multiplier"] = (

    hourly_counts["event_count"]
    /
    hourly_counts["avg_hourly_count"]

)

# prevent extreme values

hourly_counts["hour_multiplier"] = (

    hourly_counts["hour_multiplier"]
    .clip(
        lower=0.30,
        upper=3.00
    )

)

hour_multipliers = hourly_counts[
    [
        "junction",
        "hour",
        "hour_multiplier"
    ]
]

hour_multipliers.to_csv(
    "junction_hour_multipliers.csv",
    index=False
)

print(
    "Saved: junction_hour_multipliers.csv"
)

# =====================================
# WEEKDAY / WEEKEND MULTIPLIERS
# =====================================

day_counts = (

    df.groupby(
        ["junction", "day_type"]
    )
    .size()
    .reset_index(
        name="event_count"
    )

)

avg_day_counts = (

    day_counts
    .groupby("junction")["event_count"]
    .mean()
    .reset_index(
        name="avg_event_count"
    )

)

day_counts = day_counts.merge(
    avg_day_counts,
    on="junction"
)

day_counts["day_multiplier"] = (

    day_counts["event_count"]
    /
    day_counts["avg_event_count"]

)

day_counts["day_multiplier"] = (

    day_counts["day_multiplier"]
    .clip(
        lower=0.50,
        upper=2.00
    )

)

day_multipliers = day_counts[
    [
        "junction",
        "day_type",
        "day_multiplier"
    ]
]

day_multipliers.to_csv(
    "junction_daytype_multipliers.csv",
    index=False
)

print(
    "Saved: junction_daytype_multipliers.csv"
)

# =====================================
# SAMPLE OUTPUT
# =====================================

print("\nTop Junction Scores")
print(
    junction_scores.head(10)
)

print("\nHour Multipliers")
print(
    hour_multipliers.head(10)
)

print("\nDay Multipliers")
print(
    day_multipliers.head(10)
)

Saved: junction_scores.csv
Saved: junction_hour_multipliers.csv
Saved: junction_daytype_multipliers.csv

Top Junction Scores
                          junction  risk_score
175                   MekhriCircle       73.55
20               AyyappaTempleJunc       58.43
279  VeerannapalyaJunction(BEL,HO)       53.58
241          SatteliteBusStandJunc       51.50
128                     K R Circle       50.71
251                  SilkBoardJunc       48.50
291            YeshwanthpuraCircle       48.18
239                   SantheCircle       44.14
99               HebbalFlyoverJunc       44.04
132                 KIMCO Junction       44.01

Hour Multipliers
                                 junction  hour  hour_multiplier
0     17th Mn 1st Crs Aishwarya Stores Jn     3              1.0
1  27th Cross Jayanagar(Ganapathi Temple)     6              1.0
2                   28thMainJayanagarJunc     5              1.0
3                   28thMainJayanagarJunc    21              1.0
4              